Development and application of a successful Res-UNet capable of detecting and classifying cancerous bloodcells using a common blood smear.

In [3]:
# Dataset building:
# Import images and bounding box data:

import torch
from dataset_builder import idb1_dataset, idb1_collate_fn, crop_boxes_from_image

train_dataset = idb1_dataset()
val_dataset = idb1_dataset(train=False)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

Searching data\ALL_IDB\ALL_IDB1
Found 108 images and 108 centroid files.
Searching data\ALL_IDB\ALL_IDB1
Found 108 images and 108 centroid files.


c:\Users\SethM\Repositories\win_venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
c:\Users\SethM\Repositories\AI-535_GroupProject\dataset_builder.py:378: UserWarning: loadtxt: input contained no data: "data\ALL_IDB\ALL_IDB1\xyc\Im034_0.xyc"
  centroids = np.loadtxt(file_path, dtype=np.int16, delimiter='\t')
c:\Users\SethM\Repositories\AI-535_GroupProject\dataset_builder.py:378: UserWarning: loadtxt: input contained no data: "data\ALL_IDB\ALL_IDB1\xyc\Im035_0.xyc"
  centroids = np.loadtxt(file_path, dtype=np.int16, delimiter='\t')
c:\Users\SethM\Repositories\AI-535_GroupProject\dataset_builder.py:378: UserWarning: loadtxt: input contained no data: "data\ALL_IDB\ALL_IDB1\xyc\Im036_0.xyc"
  centroids = np.loadtxt(file_path, dtype=np.int16, delimiter='\t')
c:\Users\SethM\Repositories\AI-535_GroupProject\dataset_bui

In [4]:
# Visualize image and centroid-specified bounding boxes:

import torchvision.transforms.functional as F
from torchvision.utils import draw_bounding_boxes
from PIL import Image


image, target = train_dataset[1]
image_with_box = draw_bounding_boxes(image, target['boxes'], colors='red', width=3)
image_pil = F.to_pil_image(image_with_box)
image_pil.show()

In [5]:
# Build the model:

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from torchvision import models

class ConvBlock(nn.Module):
    def __init__(self, in_ch, out_ch):
        super().__init__()
        self.block = nn.Sequential(
            nn.Conv2d(in_ch, out_ch, 3, padding=1, bias=False),
            nn.BatchNorm2d(out_ch),
            nn.ReLU(inplace=True),
            nn.Conv2d(out_ch, out_ch, 3, padding=1, bias=False),
            nn.BatchNorm2d(out_ch),
            nn.ReLU(inplace=True),
        )

    def forward(self, x):
        return self.block(x)


class UpBlock(nn.Module):
    def __init__(self, in_ch, skip_ch, out_ch):
        super().__init__()
        self.up = nn.ConvTranspose2d(in_ch, in_ch // 2, kernel_size=2, stride=2)
        self.conv = ConvBlock(in_ch // 2 + skip_ch, out_ch)

    def forward(self, x, skip):
        x = self.up(x)

        # handle any off-by-one spatial differences
        if x.shape[-2:] != skip.shape[-2:]:
            x = F.interpolate(x, size=skip.shape[-2:], mode="bilinear", align_corners=False)

        x = torch.cat([x, skip], dim=1)
        return self.conv(x)


class ResUNet(nn.Module):
    def __init__(self, out_channels=1, pretrained=True):
        super().__init__()

        backbone = models.resnet34(
            weights=models.ResNet34_Weights.IMAGENET1K_V1 if pretrained else None
        )

        # Encoder
        self.input_block = nn.Sequential(
            backbone.conv1,   # 7x7, stride 2
            backbone.bn1,
            backbone.relu,
        )  # -> 64 channels
        self.maxpool = backbone.maxpool

        self.encoder1 = backbone.layer1   # 64
        self.encoder2 = backbone.layer2   # 128
        self.encoder3 = backbone.layer3   # 256
        self.encoder4 = backbone.layer4   # 512

        # Decoder
        self.center = ConvBlock(512, 512)
        self.up4 = UpBlock(512, 256, 256)
        self.up3 = UpBlock(256, 128, 128)
        self.up2 = UpBlock(128, 64, 64)
        self.up1 = UpBlock(64, 64, 64)

        self.final_up = nn.Sequential(
            nn.ConvTranspose2d(64, 32, kernel_size=2, stride=2),
            nn.ReLU(inplace=True),
            nn.Conv2d(32, out_channels, kernel_size=1)
        )

    def forward(self, x):
        # Encoder path
        x0 = self.input_block(x)      # (B, 64, H/2, W/2)
        x1 = self.maxpool(x0)         # (B, 64, H/4, W/4)
        x2 = self.encoder1(x1)        # (B, 64, H/4, W/4)
        x3 = self.encoder2(x2)        # (B,128, H/8, W/8)
        x4 = self.encoder3(x3)        # (B,256, H/16, W/16)
        x5 = self.encoder4(x4)        # (B,512, H/32, W/32)

        # Decoder path
        c = self.center(x5)
        d4 = self.up4(c, x4)
        d3 = self.up3(d4, x3)
        d2 = self.up2(d3, x2)
        d1 = self.up1(d2, x0)

        out = self.final_up(d1)

        # ensure same spatial size as input
        if out.shape[-2:] != x.shape[-2:]:
            out = F.interpolate(out, size=x.shape[-2:], mode="bilinear", align_corners=False)

        return out


In [6]:
# Calculate losses:

def dice_loss_from_logits(logits: torch.Tensor, targets: torch.Tensor, eps: float = 1e-6):
    probs = torch.sigmoid(logits)
    probs = probs.contiguous().view(probs.size(0), -1)
    targets = targets.contiguous().view(targets.size(0), -1)

    intersection = (probs * targets).sum(dim=1)
    denom = probs.sum(dim=1) + targets.sum(dim=1)

    dice = (2.0 * intersection + eps) / (denom + eps)
    return 1.0 - dice.mean()


class BCEDiceLoss(nn.Module):
    def __init__(self, bce_weight=1.0, dice_weight=1.0):
        super().__init__()
        self.bce = nn.BCEWithLogitsLoss()
        self.bce_weight = bce_weight
        self.dice_weight = dice_weight

    def forward(self, logits, targets):
        return (
            self.bce_weight * self.bce(logits, targets)
            + self.dice_weight * dice_loss_from_logits(logits, targets)
        )

In [7]:
# Train/Eval functions:

def train_one_epoch(model, loader, optimizer, criterion, device):
    """
    Train one epoch for a ResUNet-style segmentation model.

    Parameters
    ----------
    model : torch.nn.Module
        Segmentation model returning logits of shape (B,1,H,W)
    loader : DataLoader
        Must yield (images, target_dict) where target_dict contains:
          - semantic_mask: (B,1,H,W)
          - instance_masks: list of tensors
          - boxes: list of tensors
          - centroids: list of tensors
          - labels: list of tensors
    optimizer : torch.optim.Optimizer
    criterion : callable
        Loss taking (logits, target_mask)
    device : torch.device

    Returns
    -------
    metrics : dict
        epoch loss / Dice / IoU / pixel accuracy
    """
    model.train()

    total_loss = 0.0
    total_samples = 0

    total_intersection = 0.0
    total_union = 0.0
    total_pred_sum = 0.0
    total_target_sum = 0.0
    total_pixel_correct = 0.0
    total_pixels = 0.0

    for images, target in loader:
        images = images.to(device)                  # (B,C,H,W)
        masks = target["semantic_mask"].to(device)  # (B,1,H,W)

        optimizer.zero_grad()

        logits = model(images)                      # (B,1,H,W)
        if logits.shape != masks.shape:
            raise ValueError(
                f"logits shape {tuple(logits.shape)} does not match "
                f"mask shape {tuple(masks.shape)}"
            )

        loss = criterion(logits, masks)
        loss.backward()
        optimizer.step()

        B = images.size(0)
        total_loss += loss.item() * B
        total_samples += B

        # ---------------------------------------------------------
        # Metrics
        # ---------------------------------------------------------
        probs = torch.sigmoid(logits)
        preds = (probs >= 0.5).float()

        intersection = (preds * masks).sum().item()
        union = ((preds + masks) > 0).float().sum().item()

        pred_sum = preds.sum().item()
        target_sum = masks.sum().item()

        pixel_correct = (preds == masks).float().sum().item()
        pixels = masks.numel()

        total_intersection += intersection
        total_union += union
        total_pred_sum += pred_sum
        total_target_sum += target_sum
        total_pixel_correct += pixel_correct
        total_pixels += pixels

    avg_loss = total_loss / max(total_samples, 1)
    pixel_acc = total_pixel_correct / max(total_pixels, 1)

    iou = total_intersection / max(total_union, 1e-8)
    dice = (2.0 * total_intersection) / max(total_pred_sum + total_target_sum, 1e-8)

    return {
        "loss": avg_loss,
        "pixel_acc": pixel_acc,
        "iou": iou,
        "dice": dice,
    }


@torch.no_grad()
def evaluate(model, loader, criterion, device):
    """
    Evaluate one epoch for a ResUNet-style segmentation model.

    Returns
    -------
    metrics : dict
        validation/test loss + Dice + IoU + pixel accuracy
    """
    model.eval()

    total_loss = 0.0
    total_samples = 0

    total_intersection = 0.0
    total_union = 0.0
    total_pred_sum = 0.0
    total_target_sum = 0.0
    total_pixel_correct = 0.0
    total_pixels = 0.0

    for images, target in loader:
        images = images.to(device)
        masks = target["semantic_mask"].to(device)

        logits = model(images)
        if logits.shape != masks.shape:
            raise ValueError(
                f"logits shape {tuple(logits.shape)} does not match "
                f"mask shape {tuple(masks.shape)}"
            )

        loss = criterion(logits, masks)

        B = images.size(0)
        total_loss += loss.item() * B
        total_samples += B

        probs = torch.sigmoid(logits)
        preds = (probs >= 0.5).float()

        intersection = (preds * masks).sum().item()
        union = ((preds + masks) > 0).float().sum().item()

        pred_sum = preds.sum().item()
        target_sum = masks.sum().item()

        pixel_correct = (preds == masks).float().sum().item()
        pixels = masks.numel()

        total_intersection += intersection
        total_union += union
        total_pred_sum += pred_sum
        total_target_sum += target_sum
        total_pixel_correct += pixel_correct
        total_pixels += pixels

    avg_loss = total_loss / max(total_samples, 1)
    pixel_acc = total_pixel_correct / max(total_pixels, 1)

    iou = total_intersection / max(total_union, 1e-8)
    dice = (2.0 * total_intersection) / max(total_pred_sum + total_target_sum, 1e-8)

    return {
        "loss": avg_loss,
        "pixel_acc": pixel_acc,
        "iou": iou,
        "dice": dice,
    }

In [ ]:
# Training Loop:
import wandb
wandb.login()


def train_nn(batch_size: int=8, lr: float=1e-4, weight_decay: float=1e-4):

    hparams = {
        "project": f"ai535-UNet",
        "run_name": f"unet_batch-{batch_size}_lr-{lr:.5f}_weightdecay-{weight_decay:.5f}",
        "architecture": 'ResUNet34',
        "dataset": "ALL_IDB1",
        "task": "binary-detection",
        "pretrained": True,
        "loss": "dice_loss_from_logits",
        "optimizer": 'AdamW',            # options: adam, adamw, sgd
        "learning_rate": lr,
        "weight_decay": weight_decay,
        "batch_size": batch_size,
        "epochs": 75,
    }
    
    run = wandb.init(
        project=hparams["project"],
        name=hparams["run_name"],
        config=hparams,
    )

    train_loader = DataLoader(
        train_dataset,
        batch_size=batch_size,
        shuffle=True,
        num_workers=4,
        collate_fn=idb1_collate_fn,
    )

    val_loader = DataLoader(
        val_dataset,
        batch_size=batch_size,
        shuffle=False,
        num_workers=4,
        collate_fn=idb1_collate_fn,
    )

    model = ResUNet(out_channels=1, pretrained=True).to(device)
    criterion = BCEDiceLoss(bce_weight=1.0, dice_weight=1.0)
    optimizer = torch.optim.AdamW(model.parameters(), lr=1e-4, weight_decay=1e-4)

    num_epochs = 75
    best_val = float("inf")


    print('epoch | train_loss | train_dice | val_loss | val_dice | val_iou | saved')
    for epoch in range(1, num_epochs + 1):
        train_metrics = train_one_epoch(model, train_loader, optimizer, criterion, device)
        val_metrics = evaluate(model, val_loader, criterion, device)

        save_model = val_metrics['loss'] < best_val
        if save_model:
            best_val = val_metrics['loss']
            torch.save(model.state_dict(), "./models/resunet_allidb1_from_boxes.pt")

        wandb.log({
            "epoch": epoch,
            "train_loss": train_metrics['loss'],
            "train_dice": train_metrics['dice'],
            "val_loss": val_metrics['loss'],
            "val_dice": val_metrics['dice'],
            "val_iou": val_metrics['iou']
            },
            step=epoch,
            )
        print(
            f" {epoch:4d} | "
            f"{train_metrics['loss']:10.5f} | "
            f"{train_metrics['dice']:10.5f} | "
            f"{val_metrics['loss']:8.5f} | "
            f"{val_metrics['dice']:8.5f} | "
            f"{val_metrics['iou']:7.5f} | "
            f"{save_model}"
        )

        
    wandb.finish()
    print("Done.")

wandb: WARNING Calling wandb.login() after wandb.init() has no effect.


In [9]:
from itertools import product
learning_rates = [5e-5, 1e-4, 5e-4]
weight_decays = [5e-5, 1e-4, 5e-4]
batch_sizes = [4, 8, 16]

tuning_tests = list(product(batch_sizes, learning_rates, weight_decays))

for test in tuning_tests:
    train_nn(*test)

epoch | train_loss | train_dice | val_loss | val_dice | val_iou | saved
    1 |    1.63758 |    0.09988 |  1.61324 |  0.58649 | 0.41492 | True
    2 |    1.58020 |    0.48019 |  1.54900 |  0.66526 | 0.49842 | True
    3 |    1.53540 |    0.60958 |  1.47276 |  0.70147 | 0.54021 | True
    4 |    1.48741 |    0.66750 |  1.43261 |  0.75054 | 0.60069 | True


KeyboardInterrupt: 